# 17. Fusion Retrieval (BM25 + Vectors)

Combine keyword and semantic search - one of the most useful upgrades in practice.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

Dense (embedding) search understands meaning but misses exact strings - product codes, model names, rare terms - while **BM25** (keyword search) is the opposite. **Fusion retrieval** runs both and merges the rankings. It is one of the most reliable improvements you can make and is cheap, so it belongs in almost every production RAG system.

## How it works

1. Build a BM25 index over the same chunks
2. Score the query with both retrievers
3. Fuse: (A) weighted sum of normalised scores or (B) Reciprocal Rank Fusion
4. Return the fused top-k
5. Tune `alpha` / compare with each retriever alone

```text
 query --> dense scores  --\
                            +--> fusion (weighted / RRF) --> top-k
 query --> BM25 scores   --/
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

**Extra requirements for this notebook:** `pip install rank_bm25`

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
import re
import numpy as np
from rank_bm25 import BM25Okapi

tokenize = lambda s: re.findall(r"\w+", s.lower())
bm25 = BM25Okapi([tokenize(c) for c in chunks])

def dense_scores(q): return store.vecs @ embed(q)[0]
def bm25_scores(q):  return np.array(bm25.get_scores(tokenize(q)))

### Option A: weighted score fusion
Min-max normalise both score lists, then mix with weight `alpha`.

In [ ]:
def norm(x): return (x - x.min()) / (x.max() - x.min() + 1e-9)

def weighted_fusion(q, k=4, alpha=0.5):
    scores = alpha * norm(dense_scores(q)) + (1 - alpha) * norm(bm25_scores(q))
    return [{"idx": int(i), "text": chunks[i], "score": float(scores[i]), "meta": {}} for i in np.argsort(-scores)[:k]]

### Option B: Reciprocal Rank Fusion
Scores from different systems are not comparable, but ranks are. Each list adds `1 / (60 + rank)`.

In [ ]:
def rrf(q, k=4, c=60):
    fused = {}
    for scores in (dense_scores(q), bm25_scores(q)):
        for rank, i in enumerate(np.argsort(-scores)):
            fused[int(i)] = fused.get(int(i), 0) + 1 / (c + rank + 1)
    top = sorted(fused, key=fused.get, reverse=True)[:k]
    return [{"idx": i, "text": chunks[i], "score": fused[i], "meta": {}} for i in top]

query = "cross-encoder/ms-marco-MiniLM-L-6-v2"      # an exact-name query that embeddings handle poorly
for name, fn in [("dense", lambda q, k: store.search(q, k)), ("weighted", weighted_fusion), ("rrf", rrf)]:
    print("---", name); show(fn(query, 3), width=100)

In [ ]:
print(answer("What is the model name of the small widely used reranker?", [h["text"] for h in rrf("small widely used reranker model")]))

## Where each retriever wins
Exact-term queries favour BM25; paraphrased queries favour dense search.

In [ ]:
import pandas as pd
tests = {
    "ms-marco-MiniLM-L-6-v2": "cross-encoder/ms-marco-MiniLM-L-6-v2 is a small",
    "HNSW": "HNSW builds a layered graph",
    "how do I make results trustworthy": "Grounding techniques include",
    "what is the 384 thing": "384-dimensional vectors",
}
rows = []
for q, ev in tests.items():
    row = {"query": q}
    for name, fn in {"dense": lambda q, k: store.search(q, k), "bm25": lambda q, k: [{"text": chunks[i]} for i in np.argsort(-bm25_scores(q))[:k]], "rrf": rrf}.items():
        row[name] = evidence_rank(fn(q, 5), ev)
    rows.append(row)
pd.DataFrame(rows).set_index("query")

## Evaluate on the golden set

In [ ]:
bm25_search = lambda q, k=3: [{"text": chunks[i], "idx": int(i)} for i in np.argsort(-bm25_scores(q))[:k]]
compare_retrievers({"dense": store.search, "BM25": bm25_search, "weighted (0.5)": weighted_fusion, "RRF": rrf}, k=3)

## Tune `alpha` in the weighted fusion

In [ ]:
rows = []
for a in [0.0, 0.25, 0.5, 0.75, 1.0]:
    r = evaluate_retriever(lambda q, k, a=a: weighted_fusion(q, k, alpha=a), k=3)
    rows.append({"alpha (1 = all dense)": a, "hit@3": r["hit_rate@3"], "MRR": r["MRR"]})
pd.DataFrame(rows).set_index("alpha (1 = all dense)")

## Why RRF is robust
BM25 scores are unbounded; cosine is in [-1, 1]. Mixing raw scores is meaningless, so RRF uses only **ranks**.

In [ ]:
q = "reciprocal rank fusion constant"
print("raw BM25 top scores :", np.sort(bm25_scores(q))[::-1][:3].round(2))
print("raw cosine top score:", np.sort(dense_scores(q))[::-1][:3].round(2))

## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Handles both exact terms and paraphrases | Two indexes to maintain |
| Cheap (BM25 is fast, no GPU) | Weighted fusion needs normalisation and tuning |
| Large recall gains in practice | Tokenisation choices affect BM25 |

## When to use it

Almost always - especially for technical, product or legal content with identifiers and jargon.

## Experiments to try

- Add stop-word removal and stemming to the BM25 tokenizer.
- Try RRF constants 10, 60, 100.
- Add a third retriever (e.g. a different embedding model) to the fusion.

## Key takeaways

- Dense + sparse beats either alone.
- Prefer RRF when score scales differ.
- Part of the must-do core.

---
**Next:** [18. Reranking with a Cross-Encoder](../18_reranking/reranking.ipynb)